<a href="https://www.kaggle.com/code/vidushigupta1/brainstrokeprediction?scriptVersionId=330988651" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [3]:
# ============================================================
# CELL 1 — Install & Imports
# ============================================================
!pip install shap --quiet

import os, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import shap

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, roc_auc_score, roc_curve,
    precision_recall_curve, confusion_matrix, classification_report,
    average_precision_score
)
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# Output directory for all 300 dpi figures
os.makedirs('/kaggle/working/figures', exist_ok=True)
os.makedirs('/kaggle/working/models',  exist_ok=True)

✅ Using device: cuda


In [4]:
# ============================================================
# CELL 2 — Load Data & EDA
# ============================================================
data = pd.read_csv('/kaggle/input/cerebral-stroke-predictionimbalaced-dataset/dataset.csv')
print("Shape:", data.shape)
print("\nClass distribution:")
print(data['stroke'].value_counts())
print("\nMissing values:")
print(data.isnull().sum())
data.head()

Shape: (43400, 12)

Class distribution:
stroke
0    42617
1      783
Name: count, dtype: int64

Missing values:
id                       0
gender                   0
age                      0
hypertension             0
heart_disease            0
ever_married             0
work_type                0
Residence_type           0
avg_glucose_level        0
bmi                   1462
smoking_status       13292
stroke                   0
dtype: int64


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,30669,Male,3.0,0,0,No,children,Rural,95.12,18.0,NaN,0
1,30468,Male,58.0,1,0,Yes,Private,Urban,87.96,39.2,never smoked,0
2,16523,Female,8.0,0,0,No,Private,Urban,110.89,17.6,NaN,0
3,56543,Female,70.0,0,0,Yes,Private,Rural,69.04,35.9,formerly smoked,0
4,46136,Male,14.0,0,0,No,Never_worked,Rural,161.28,19.1,NaN,0


In [5]:
# ============================================================
# CELL 3 — Preprocessing
# ============================================================
df = data.copy()
df.drop(columns=['id'], inplace=True)

FEATURE_COLS = [c for c in df.columns if c != 'stroke']

# Label-encode categoricals — fit on FULL data so encoder covers all categories
cat_cols = df.select_dtypes(include='object').columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Impute missing values (median — safer than mean for skewed data)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
df[FEATURE_COLS] = imputer.fit_transform(df[FEATURE_COLS])

X_all = df[FEATURE_COLS].values
y_all = df['stroke'].values

print(f"✅ Features shape: {X_all.shape}")
print(f"✅ Positive (stroke) samples: {y_all.sum()} / {len(y_all)}")

✅ Features shape: (43400, 10)
✅ Positive (stroke) samples: 783 / 43400


In [6]:
# ============================================================
# CELL 4 — Conditional GAN Definition
# ============================================================
N_FEATURES = X_all.shape[1]   # 11
NOISE_DIM   = 64
EMBED_DIM   = 8

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(2, EMBED_DIM)
        self.net = nn.Sequential(
            nn.Linear(NOISE_DIM + EMBED_DIM, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256),                    nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),                    nn.LeakyReLU(0.2),
            nn.Linear(128, N_FEATURES),             nn.Tanh()
        )
    def forward(self, z, labels):
        x = torch.cat([z, self.label_emb(labels)], dim=1)
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(2, EMBED_DIM)
        self.net = nn.Sequential(
            nn.Linear(N_FEATURES + EMBED_DIM, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 128),                     nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, x, labels):
        inp = torch.cat([x, self.label_emb(labels)], dim=1)
        return self.net(inp)

def train_cgan(X_minority, epochs=300, batch_size=64, lr=2e-4, tag=''):
    """Train cGAN on minority class samples and return generator."""
    X_t = torch.FloatTensor(X_minority).to(device)
    labels_ones = torch.ones(len(X_t), dtype=torch.long).to(device)

    G = Generator().to(device)
    D = Discriminator().to(device)

    opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCEWithLogitsLoss()

    g_losses, d_losses = [], []
    dataset = TensorDataset(X_t, labels_ones)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    print(f"\n🔵 Training cGAN {tag}...")
    for epoch in range(1, epochs + 1):
        g_ep, d_ep = [], []
        for real_x, real_l in loader:
            bs = real_x.size(0)
            real_lbl = torch.ones(bs, 1).to(device)
            fake_lbl = torch.zeros(bs, 1).to(device)
            fake_labels_batch = torch.ones(bs, dtype=torch.long).to(device)

            # Train D
            z = torch.randn(bs, NOISE_DIM).to(device)
            fake_x = G(z, fake_labels_batch).detach()
            d_real = D(real_x, real_l)
            d_fake = D(fake_x, fake_labels_batch)
            d_loss = (criterion(d_real, real_lbl) + criterion(d_fake, fake_lbl)) / 2
            opt_D.zero_grad(); d_loss.backward(); opt_D.step()

            # Train G
            z = torch.randn(bs, NOISE_DIM).to(device)
            fake_x = G(z, fake_labels_batch)
            g_loss = criterion(D(fake_x, fake_labels_batch), real_lbl)
            opt_G.zero_grad(); g_loss.backward(); opt_G.step()

            g_ep.append(g_loss.item()); d_ep.append(d_loss.item())

        g_losses.append(np.mean(g_ep))
        d_losses.append(np.mean(d_ep))
        if epoch % 50 == 0:
            print(f"  Epoch {epoch:>3}/{epochs} | G Loss: {g_losses[-1]:.4f} | D Loss: {d_losses[-1]:.4f}")

    # Plot cGAN loss
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(g_losses, label='Generator Loss',     color='#00b4d8')
    ax.plot(d_losses, label='Discriminator Loss', color='#f77f7f')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'cGAN Training Loss {tag}'); ax.legend()
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/cgan_loss_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"✅ cGAN training complete! {tag}")
    return G

def generate_synthetic(G, n_samples, scaler=None):
    G.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, NOISE_DIM).to(device)
        labels = torch.ones(n_samples, dtype=torch.long).to(device)
        syn = G(z, labels).cpu().numpy()
    # syn is in [-1,1] (Tanh). Inverse-scale if scaler provided
    if scaler is not None:
        syn = scaler.inverse_transform(syn)
    return syn

In [7]:
# ============================================================
# CELL 5 — Model Architecture
# ============================================================

class CNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, stride=1, padding=1)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.act   = nn.GELU()
        self.proj  = nn.Conv1d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        residual = self.proj(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.act(x + residual)

class BiLSTMBlock(nn.Module):
    def __init__(self, input_size=32, hidden=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden, num_layers=num_layers,
                               batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.dropout(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=128, nhead=4, dim_ff=256, num_layers=2, dropout=0.3):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=dim_ff,
                                               dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
    def forward(self, x):
        return self.encoder(x)

class HybridStrokeModel(nn.Module):
    """
    Full proposed model: CNN → BiLSTM → proj → Transformer → Classifier
    use_bilstm / use_transformer flags enable ablation variants.
    """
    def __init__(self, n_features=11, use_bilstm=True, use_transformer=True, dropout=0.3):
        super().__init__()
        self.use_bilstm     = use_bilstm
        self.use_transformer = use_transformer

        # CNN backbone
        self.cnn = nn.Sequential(
            CNNBlock(1, 16),
            CNNBlock(16, 32)
        )
        cnn_out_ch = 32  # channels after CNN

        # BiLSTM
        if use_bilstm:
            self.bilstm = BiLSTMBlock(input_size=cnn_out_ch, hidden=64, num_layers=2, dropout=dropout)
            lstm_out = 128  # bidirectional → 64*2
        else:
            lstm_out = cnn_out_ch  # will use CNN channels directly

        # Projection to d_model=128
        self.proj = nn.Linear(lstm_out, 128)

        # Transformer
        if use_transformer:
            self.transformer = TransformerBlock(d_model=128, nhead=4, dim_ff=256,
                                                num_layers=2, dropout=dropout)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, 1, n_features)
        x = self.cnn(x)               # (B, 32, n_features)

        if self.use_bilstm:
            x = x.permute(0, 2, 1)   # (B, n_features, 32)
            x = self.bilstm(x)        # (B, n_features, 128)
        else:
            x = x.permute(0, 2, 1)   # (B, n_features, 32)

        x = self.proj(x)              # (B, n_features, 128)

        if self.use_transformer:
            x = self.transformer(x)   # (B, n_features, 128)

        x = x.mean(dim=1)             # Global avg pool → (B, 128)
        return self.classifier(x)     # (B, 1)


def build_model(variant='full'):
    """Factory for all ablation variants."""
    configs = {
        'cnn_only':       dict(use_bilstm=False, use_transformer=False),
        'cnn_bilstm':     dict(use_bilstm=True,  use_transformer=False),
        'cnn_transformer':dict(use_bilstm=False, use_transformer=True),
        'full':           dict(use_bilstm=True,  use_transformer=True),
    }
    cfg = configs.get(variant, configs['full'])
    return HybridStrokeModel(n_features=N_FEATURES, **cfg).to(device)

# Quick architecture check
m = build_model('full')
print("Model architecture:"); print(m)
total_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")
del m

Model architecture:
HybridStrokeModel(
  (cnn): Sequential(
    (0): CNNBlock(
      (conv1): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (proj): Conv1d(1, 16, kernel_size=(1,), stride=(1,))
    )
    (1): CNNBlock(
      (conv1): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (proj): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
    )
  )
  (bilstm): BiLSTMBl

In [8]:
# ============================================================
# CELL 6 — Focal Loss + Training & Evaluation Utilities (FIXED)
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):  # alpha=0.75 heavily weights minority
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt   = torch.exp(-bce)
        # alpha for positive (stroke=1), (1-alpha) for negative
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - pt) ** self.gamma * bce
        return loss.mean()


def make_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size=64):
    def to_tensor(X, y):
        Xt = torch.FloatTensor(X).unsqueeze(1)
        yt = torch.FloatTensor(y)
        return TensorDataset(Xt, yt)
    tr_dl  = DataLoader(to_tensor(X_tr, y_tr),  batch_size=batch_size, shuffle=True,  drop_last=False)
    val_dl = DataLoader(to_tensor(X_val, y_val), batch_size=batch_size, shuffle=False, drop_last=False)
    te_dl  = DataLoader(to_tensor(X_te, y_te),   batch_size=batch_size, shuffle=False, drop_last=False)
    return tr_dl, val_dl, te_dl


def find_best_threshold(y_true, y_probs):
    """Find threshold that maximises F1 on the given set."""
    from sklearn.metrics import f1_score
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (y_probs >= t).astype(int)
        f = f1_score(y_true, preds, zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_t  = t
    return best_t, best_f1


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out  = model(X_b).squeeze(1)
        loss = criterion(out, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for X_b, y_b in loader:
        X_b = X_b.to(device)
        out = model(X_b).squeeze(1)
        all_probs.append(torch.sigmoid(out).cpu())
        all_labels.append(y_b)
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    return probs, labels


def compute_metrics(y_true, y_probs, threshold=0.5):
    preds = (y_probs >= threshold).astype(int)
    return {
        'probs':  y_probs,
        'preds':  preds,
        'labels': y_true,
        'acc':    accuracy_score(y_true, preds),
        'prec':   precision_score(y_true, preds, zero_division=0),
        'rec':    recall_score(y_true, preds, zero_division=0),
        'f1':     f1_score(y_true, preds, zero_division=0),
        'kappa':  cohen_kappa_score(y_true, preds),
        'auc':    roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.0,
    }


def full_train(model, X_tr, y_tr, X_val, y_val, X_te, y_te,
               epochs=50, lr=1e-3, use_focal=True, save_path=None, tag=''):

    tr_dl, val_dl, te_dl = make_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te)

    # ── Weighted BCE as fallback + Focal ──────────────────────────────
    if use_focal:
        criterion = FocalLoss(alpha=0.75, gamma=2.0)
    else:
        # Class-weighted BCE — critical for imbalanced test sets
        pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)]).to(device)
        criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_f1, best_state  = 0.0, None
    best_threshold       = 0.5
    train_losses, val_losses = [], []
    val_f1s, val_aucs        = [], []

    t_start = time.time()

    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, tr_dl, optimizer, criterion)
        scheduler.step()

        val_probs, val_labels = get_probs(model, val_dl)

        # Find best threshold on validation set each epoch
        if len(np.unique(val_labels)) > 1:
            t_ep, vf1 = find_best_threshold(val_labels, val_probs)
            vauc = roc_auc_score(val_labels, val_probs)
        else:
            t_ep, vf1, vauc = 0.5, 0.0, 0.5

        val_loss = nn.functional.binary_cross_entropy(
            torch.FloatTensor(val_probs),
            torch.FloatTensor(val_labels.astype(float))
        ).item()

        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        val_f1s.append(vf1)
        val_aucs.append(vauc)

        if vf1 > best_f1:
            best_f1        = vf1
            best_threshold = t_ep
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}

        if ep % 10 == 0:
            print(f"  Ep {ep:>3}/{epochs} | TrLoss:{tr_loss:.4f} | "
                  f"ValLoss:{val_loss:.4f} | ValF1:{vf1:.4f} | ValAUC:{vauc:.4f} | Thr:{t_ep:.2f}")

    training_time = time.time() - t_start

    # Load best checkpoint
    model.load_state_dict(best_state)
    if save_path:
        torch.save({'state_dict': best_state, 'threshold': best_threshold}, save_path)
        print(f"  ✅ Model saved → {save_path}  (best threshold: {best_threshold:.2f})")

    # Inference time
    model.eval()
    te_tensor = torch.FloatTensor(X_te).unsqueeze(1).to(device)
    t0 = time.time()
    with torch.no_grad():
        _ = model(te_tensor)
    inf_time_ms = (time.time() - t0) * 1000

    # Model size
    tmp_path = '/kaggle/working/models/_tmp_size.pt'
    torch.save(model.state_dict(), tmp_path)
    model_size_mb = os.path.getsize(tmp_path) / 1e6

    # Test evaluation with best threshold from validation
    te_probs, te_labels = get_probs(model, te_dl)
    te_res = compute_metrics(te_labels, te_probs, threshold=best_threshold)

    print(f"\n{'='*60}")
    print(f"  [{tag}] FINAL TEST RESULTS  (threshold={best_threshold:.2f})")
    print(f"{'='*60}")
    print(f"  Accuracy  : {te_res['acc']:.4f}")
    print(f"  Precision : {te_res['prec']:.4f}")
    print(f"  Recall    : {te_res['rec']:.4f}")
    print(f"  F1 Score  : {te_res['f1']:.4f}")
    print(f"  Kappa     : {te_res['kappa']:.4f}")
    print(f"  ROC-AUC   : {te_res['auc']:.4f}")
    print(f"  Train Time: {training_time:.1f}s")
    print(f"  Infer Time: {inf_time_ms:.2f}ms")
    print(f"  Model Size: {model_size_mb:.2f}MB")
    print(classification_report(te_labels, te_res['preds'],
                                target_names=['No Stroke','Stroke']))

    te_res.update({
        'train_time':    training_time,
        'inf_time_ms':   inf_time_ms,
        'model_size_mb': model_size_mb,
        'train_losses':  train_losses,
        'val_losses':    val_losses,
        'val_f1s':       val_f1s,
        'val_aucs':      val_aucs,
        'threshold':     best_threshold,
        'tag':           tag,
    })
    return te_res

In [9]:
# ============================================================
# CELL 7 — Figure Saving Utilities (300 dpi)
# ============================================================

FEATURE_NAMES = None  # will be set after preprocessing

def save_confusion_matrix(y_true, y_pred, tag):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Stroke','Stroke'],
                yticklabels=['No Stroke','Stroke'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {tag}')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/cm_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

def save_loss_curve(train_losses, val_losses, tag):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label='Train Loss', color='#1a56db')
    ax.plot(val_losses,   label='Val Loss',   color='#f97316')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Focal Loss')
    ax.set_title(f'Train vs Val Loss — {tag}')
    ax.legend(); plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/loss_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

def save_val_metrics_curve(val_f1s, val_aucs, tag):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(val_f1s,  label='Val F1',  color='#0ea5e9')
    ax.plot(val_aucs, label='Val AUC', color='#22c55e')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
    ax.set_title(f'Validation Metrics — {tag}')
    ax.set_ylim(0, 1.05); ax.legend(); plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/val_metrics_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

def save_roc_curve(y_true, y_probs, tag):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_val = roc_auc_score(y_true, y_probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, color='#1a56db', lw=2, label=f'AUC = {auc_val:.4f}')
    ax.plot([0,1],[0,1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {tag}'); ax.legend(); plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/roc_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

def save_pr_curve(y_true, y_probs, tag):
    prec, rec, _ = precision_recall_curve(y_true, y_probs)
    ap = average_precision_score(y_true, y_probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, color='#f97316', lw=2, label=f'AP = {ap:.4f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'Precision-Recall Curve — {tag}'); ax.legend(); plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/pr_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

def save_all_figures(res, tag):
    """Save all 4 figures for a given variant result dict."""
    save_confusion_matrix(res['labels'], res['preds'], tag)
    save_loss_curve(res['train_losses'], res['val_losses'], tag)
    save_val_metrics_curve(res['val_f1s'], res['val_aucs'], tag)
    save_roc_curve(res['labels'], res['probs'], tag)
    save_pr_curve(res['labels'], res['probs'], tag)
    print(f"✅ All figures saved for [{tag}]")

In [10]:
# ============================================================
# CELL 8 — VARIANT A: No Balancing (Imbalanced Baseline)
# ============================================================
print("\n" + "="*60)
print("  VARIANT A — Original Dataset, No Balancing")
print("="*60)

# Scale raw data
scaler_A = StandardScaler()
X_scaled_A = scaler_A.fit_transform(X_all)
FEATURE_NAMES = FEATURE_COLS  # save for GradCAM/SHAP

# Split (stratified to preserve 1.8% stroke ratio)
X_trA, X_tmpA, y_trA, y_tmpA = train_test_split(
    X_scaled_A, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_vA, X_teA, y_vA, y_teA = train_test_split(
    X_tmpA, y_tmpA, test_size=0.50, stratify=y_tmpA, random_state=SEED)

print(f"Train: {len(y_trA)} | Val: {len(y_vA)} | Test: {len(y_teA)}")
print(f"Train stroke: {y_trA.sum()} / {len(y_trA)}")

model_A = build_model('full')
res_A = full_train(model_A, X_trA, y_trA, X_vA, y_vA, X_teA, y_teA,
                   epochs=50, tag='VariantA_NoBalance',
                   save_path='/kaggle/working/models/model_variantA.pt')
save_all_figures(res_A, 'VariantA_NoBalance')


  VARIANT A — Original Dataset, No Balancing
Train: 30380 | Val: 6510 | Test: 6510
Train stroke: 548 / 30380
  Ep  10/50 | TrLoss:0.0119 | ValLoss:0.2794 | ValF1:0.1234 | ValAUC:0.8329 | Thr:0.37
  Ep  20/50 | TrLoss:0.0116 | ValLoss:0.2372 | ValF1:0.1257 | ValAUC:0.8314 | Thr:0.31
  Ep  30/50 | TrLoss:0.0115 | ValLoss:0.2464 | ValF1:0.1312 | ValAUC:0.8335 | Thr:0.34
  Ep  40/50 | TrLoss:0.0113 | ValLoss:0.2271 | ValF1:0.1309 | ValAUC:0.8353 | Thr:0.34
  Ep  50/50 | TrLoss:0.0112 | ValLoss:0.2303 | ValF1:0.1330 | ValAUC:0.8368 | Thr:0.34
  ✅ Model saved → /kaggle/working/models/model_variantA.pt  (best threshold: 0.34)

  [VariantA_NoBalance] FINAL TEST RESULTS  (threshold=0.34)
  Accuracy  : 0.9393
  Precision : 0.0888
  Recall    : 0.2564
  F1 Score  : 0.1319
  Kappa     : 0.1081
  ROC-AUC   : 0.8475
  Train Time: 311.9s
  Infer Time: 10.54ms
  Model Size: 1.81MB
              precision    recall  f1-score   support

   No Stroke       0.99      0.95      0.97      6393
      Stroke

In [11]:
# ============================================================
# CELL 9 — VARIANT B: Full Dataset cGAN Balance (original approach)
# ============================================================
print("\n" + "="*60)
print("  VARIANT B — Complete Dataset Balanced (cGAN, full data)")
print("="*60)

# Scale
scaler_B = StandardScaler()
X_scaled_B = scaler_B.fit_transform(X_all)

# Train cGAN on ALL minority samples (original approach)
X_min_B = X_scaled_B[y_all == 1]
n_to_gen_B = (y_all == 0).sum() - (y_all == 1).sum()  # how many to generate

G_B = train_cgan(X_min_B, epochs=300, tag='VarB')
X_syn_B = generate_synthetic(G_B, n_to_gen_B)

X_bal_B = np.vstack([X_scaled_B, X_syn_B])
y_bal_B = np.hstack([y_all, np.ones(n_to_gen_B, dtype=int)])
print(f"✅ Balanced dataset: Total={len(y_bal_B)} | Stroke={y_bal_B.sum()} | No-stroke={(y_bal_B==0).sum()}")

# Bar chart before/after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['No Stroke','Stroke'], [(y_all==0).sum(), (y_all==1).sum()], color=['#00b4d8','#f77f7f'])
axes[0].set_title('Before cGAN'); axes[0].set_ylabel('Count')
axes[1].bar(['No Stroke','Stroke'], [(y_bal_B==0).sum(), (y_bal_B==1).sum()], color=['#00b4d8','#f77f7f'])
axes[1].set_title('After cGAN')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/cgan_balance_VarB.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()

# Split (after balancing — this is the original protocol with leakage note)
X_trB, X_tmpB, y_trB, y_tmpB = train_test_split(
    X_bal_B, y_bal_B, test_size=0.30, stratify=y_bal_B, random_state=SEED)
X_vB, X_teB, y_vB, y_teB = train_test_split(
    X_tmpB, y_tmpB, test_size=0.50, stratify=y_tmpB, random_state=SEED)
print(f"Train: {len(y_trB)} | Val: {len(y_vB)} | Test: {len(y_teB)}")

model_B = build_model('full')
res_B = full_train(model_B, X_trB, y_trB, X_vB, y_vB, X_teB, y_teB,
                   epochs=50, tag='VariantB_FullBalance',
                   save_path='/kaggle/working/models/model_variantB.pt')
save_all_figures(res_B, 'VariantB_FullBalance')


  VARIANT B — Complete Dataset Balanced (cGAN, full data)

🔵 Training cGAN VarB...
  Epoch  50/300 | G Loss: 1.3080 | D Loss: 0.4030
  Epoch 100/300 | G Loss: 1.8738 | D Loss: 0.2594
  Epoch 150/300 | G Loss: 2.4578 | D Loss: 0.1434
  Epoch 200/300 | G Loss: 3.0316 | D Loss: 0.1065
  Epoch 250/300 | G Loss: 3.2017 | D Loss: 0.0841
  Epoch 300/300 | G Loss: 3.5418 | D Loss: 0.0607
✅ cGAN training complete! VarB
✅ Balanced dataset: Total=85234 | Stroke=42617 | No-stroke=42617
Train: 59663 | Val: 12785 | Test: 12786
  Ep  10/50 | TrLoss:0.0065 | ValLoss:0.1364 | ValF1:0.9894 | ValAUC:0.9966 | Thr:0.55
  Ep  20/50 | TrLoss:0.0062 | ValLoss:0.1304 | ValF1:0.9895 | ValAUC:0.9969 | Thr:0.41
  Ep  30/50 | TrLoss:0.0060 | ValLoss:0.1365 | ValF1:0.9894 | ValAUC:0.9969 | Thr:0.43
  Ep  40/50 | TrLoss:0.0058 | ValLoss:0.1299 | ValF1:0.9895 | ValAUC:0.9968 | Thr:0.44
  Ep  50/50 | TrLoss:0.0057 | ValLoss:0.1186 | ValF1:0.9895 | ValAUC:0.9968 | Thr:0.48
  ✅ Model saved → /kaggle/working/models/mode

In [12]:
Z# ============================================================
# CELL 10 — VARIANT C: Leakage-Free (FIXED)
# ============================================================
print("\n" + "="*60)
print("  VARIANT C — Leakage-Free: Balance Train Only")
print("="*60)

scaler_C = StandardScaler()
X_scaled_C = scaler_C.fit_transform(X_all)

# Step 1: Split FIRST
X_trC, X_tmpC, y_trC, y_tmpC = train_test_split(
    X_scaled_C, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_vC, X_teC, y_vC, y_teC = train_test_split(
    X_tmpC, y_tmpC, test_size=0.50, stratify=y_tmpC, random_state=SEED)

print(f"Before balancing — Train: {len(y_trC)} (stroke={y_trC.sum()}) | "
      f"Val: {len(y_vC)} (stroke={y_vC.sum()}) | Test: {len(y_teC)} (stroke={y_teC.sum()})")

# Step 2: cGAN only on train minority
X_min_C    = X_trC[y_trC == 1]
n_majority = (y_trC == 0).sum()
n_minority = (y_trC == 1).sum()
n_to_gen_C = n_majority - n_minority
print(f"Generating {n_to_gen_C} synthetic stroke samples...")

G_C      = train_cgan(X_min_C, epochs=300, tag='VarC')
X_syn_C  = generate_synthetic(G_C, n_to_gen_C)

# Clip synthetic samples to same range as real training data
X_syn_C  = np.clip(X_syn_C, X_trC.min(axis=0), X_trC.max(axis=0))

# Step 3: Append only to train
X_trC_bal = np.vstack([X_trC, X_syn_C])
y_trC_bal = np.hstack([y_trC, np.ones(n_to_gen_C, dtype=int)])

# Shuffle
shuffle_idx   = np.random.permutation(len(X_trC_bal))
X_trC_bal     = X_trC_bal[shuffle_idx]
y_trC_bal     = y_trC_bal[shuffle_idx]

print(f"After  balancing — Train: {len(y_trC_bal)} "
      f"(stroke={y_trC_bal.sum()}) | Val/Test: original imbalanced")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['No Stroke','Stroke'], [n_majority, n_minority], color=['#00b4d8','#f77f7f'])
axes[0].set_title('Train Before cGAN (Variant C)'); axes[0].set_ylabel('Count')
axes[1].bar(['No Stroke','Stroke'], [(y_trC_bal==0).sum(),(y_trC_bal==1).sum()], color=['#00b4d8','#f77f7f'])
axes[1].set_title('Train After cGAN (Variant C)')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/cgan_balance_VarC.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()

model_C = build_model('full')
res_C   = full_train(model_C,
                     X_trC_bal, y_trC_bal,
                     X_vC, y_vC,
                     X_teC, y_teC,
                     epochs=50, tag='VariantC_TrainOnly',
                     save_path='/kaggle/working/models/model_variantC.pt')
save_all_figures(res_C, 'VariantC_TrainOnly')
print("✅ Variant C complete.")


  VARIANT C — Leakage-Free: Balance Train Only
Before balancing — Train: 30380 (stroke=548) | Val: 6510 (stroke=118) | Test: 6510 (stroke=117)
Generating 29284 synthetic stroke samples...

🔵 Training cGAN VarC...
  Epoch  50/300 | G Loss: 1.1589 | D Loss: 0.4749
  Epoch 100/300 | G Loss: 1.5266 | D Loss: 0.3412
  Epoch 150/300 | G Loss: 1.7300 | D Loss: 0.2893
  Epoch 200/300 | G Loss: 2.2677 | D Loss: 0.1770
  Epoch 250/300 | G Loss: 2.7975 | D Loss: 0.1084
  Epoch 300/300 | G Loss: 2.8368 | D Loss: 0.0995
✅ cGAN training complete! VarC
After  balancing — Train: 59664 (stroke=29832) | Val/Test: original imbalanced
  Ep  10/50 | TrLoss:0.0066 | ValLoss:0.2275 | ValF1:0.1258 | ValAUC:0.8352 | Thr:0.33
  Ep  20/50 | TrLoss:0.0063 | ValLoss:0.2337 | ValF1:0.1307 | ValAUC:0.8347 | Thr:0.35
  Ep  30/50 | TrLoss:0.0060 | ValLoss:0.2514 | ValF1:0.1239 | ValAUC:0.8315 | Thr:0.33
  Ep  40/50 | TrLoss:0.0058 | ValLoss:0.2431 | ValF1:0.1213 | ValAUC:0.8285 | Thr:0.35
  Ep  50/50 | TrLoss:0.0056 

In [13]:
# ============================================================
# CELL 11 — Ablation Study (same Variant C balanced train split)
# ============================================================
print("\n" + "="*60)
print("  ABLATION STUDY")
print("="*60)

ABLATION_CONFIGS = {
    'Abl1_CNN_Only':        {'variant': 'cnn_only',        'use_focal': True},
    'Abl2_CNN_BiLSTM':      {'variant': 'cnn_bilstm',      'use_focal': True},
    'Abl3_CNN_Transformer': {'variant': 'cnn_transformer', 'use_focal': True},
    'Abl4_Full_Proposed':   {'variant': 'full',            'use_focal': True},
    'Abl5_Full_NoBCE':      {'variant': 'full',            'use_focal': False},
}

ablation_results = {}

for name, cfg in ABLATION_CONFIGS.items():
    print(f"\n▶ Training: {name}")
    set_seed()
    model = build_model(cfg['variant'])
    res = full_train(
        model,
        X_trC_bal, y_trC_bal,
        X_vC, y_vC,
        X_teC, y_teC,
        epochs=50,
        use_focal=cfg['use_focal'],
        tag=name,
        save_path=f'/kaggle/working/models/{name}.pt'
    )
    ablation_results[name] = res
    save_all_figures(res, name)

print("\n✅ All ablation variants complete.")


  ABLATION STUDY

▶ Training: Abl1_CNN_Only
  Ep  10/50 | TrLoss:0.0067 | ValLoss:0.2303 | ValF1:0.1150 | ValAUC:0.8119 | Thr:0.36
  Ep  20/50 | TrLoss:0.0064 | ValLoss:0.2374 | ValF1:0.1163 | ValAUC:0.8214 | Thr:0.31
  Ep  30/50 | TrLoss:0.0062 | ValLoss:0.2463 | ValF1:0.1190 | ValAUC:0.8226 | Thr:0.33
  Ep  40/50 | TrLoss:0.0059 | ValLoss:0.2297 | ValF1:0.1276 | ValAUC:0.8311 | Thr:0.31
  Ep  50/50 | TrLoss:0.0058 | ValLoss:0.2252 | ValF1:0.1249 | ValAUC:0.8276 | Thr:0.31
  ✅ Model saved → /kaggle/working/models/Abl1_CNN_Only.pt  (best threshold: 0.34)

  [Abl1_CNN_Only] FINAL TEST RESULTS  (threshold=0.34)
  Accuracy  : 0.9147
  Precision : 0.0788
  Recall    : 0.3504
  F1 Score  : 0.1287
  Kappa     : 0.1024
  ROC-AUC   : 0.8341
  Train Time: 271.2s
  Infer Time: 1.34ms
  Model Size: 0.09MB
              precision    recall  f1-score   support

   No Stroke       0.99      0.93      0.96      6393
      Stroke       0.08      0.35      0.13       117

    accuracy                 

In [14]:
# ============================================================
# CELL 12 — 5-Fold Cross-Validation + Wilcoxon (FIXED)
# ============================================================
print("\n" + "="*60)
print("  5-FOLD CROSS-VALIDATION + STATISTICAL TESTS")
print("="*60)

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

kfold_scores = {name: {'acc':[],'f1':[],'kappa':[],'auc':[]} for name in ABLATION_CONFIGS}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_trC_bal, y_trC_bal)):
    print(f"\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────")
    Xf_tr,  Xf_val  = X_trC_bal[tr_idx],  X_trC_bal[val_idx]
    yf_tr,  yf_val  = y_trC_bal[tr_idx],  y_trC_bal[val_idx]

    for name, cfg in ABLATION_CONFIGS.items():
        set_seed()
        m = build_model(cfg['variant'])
        res = full_train(m, Xf_tr, yf_tr, Xf_val, yf_val,
                         X_teC, y_teC,                      # always eval on held-out test
                         epochs=30, use_focal=cfg['use_focal'],
                         tag=f'{name}_fold{fold+1}')
        kfold_scores[name]['acc'].append(res['acc'])
        kfold_scores[name]['f1'].append(res['f1'])
        kfold_scores[name]['kappa'].append(res['kappa'])
        kfold_scores[name]['auc'].append(res['auc'])
        print(f"  {name}: F1={res['f1']:.4f}  AUC={res['auc']:.4f}  Thr={res['threshold']:.2f}")

# ── Summary ─────────────────────────────────────────────────
print("\n" + "="*70)
print(f"  {'Model':<30} | {'F1 Mean±Std':^20} | {'AUC Mean±Std':^20} | {'Kappa':^20}")
print("="*70)
for name in ABLATION_CONFIGS:
    f1s  = kfold_scores[name]['f1']
    aucs = kfold_scores[name]['auc']
    kaps = kfold_scores[name]['kappa']
    print(f"  {name:<30} | {np.mean(f1s):.4f} ± {np.std(f1s):.4f}  "
          f"| {np.mean(aucs):.4f} ± {np.std(aucs):.4f}  "
          f"| {np.mean(kaps):.4f} ± {np.std(kaps):.4f}")

# ── Wilcoxon ─────────────────────────────────────────────────
print("\n── Wilcoxon Signed-Rank Test (F1): Proposed vs Each Ablation ──")
proposed_f1 = kfold_scores['Abl4_Full_Proposed']['f1']
for name in ABLATION_CONFIGS:
    if name == 'Abl4_Full_Proposed': continue
    scores = kfold_scores[name]['f1']
    diff   = np.array(proposed_f1) - np.array(scores)
    if np.all(diff == 0):
        print(f"  Proposed vs {name}: no difference (p=1.0)")
        continue
    try:
        stat, p = wilcoxon(proposed_f1, scores)
        sig = "✅ p<0.05 Significant" if p < 0.05 else "❌ Not Significant"
        print(f"  Proposed vs {name:<30}: stat={stat:.4f}, p={p:.4f}  {sig}")
    except Exception as e:
        print(f"  Proposed vs {name}: {e}")


  5-FOLD CROSS-VALIDATION + STATISTICAL TESTS

── Fold 1/5 ──────────────────────────────
  Ep  10/30 | TrLoss:0.0067 | ValLoss:0.1427 | ValF1:0.9904 | ValAUC:0.9970 | Thr:0.51
  Ep  20/30 | TrLoss:0.0062 | ValLoss:0.1203 | ValF1:0.9917 | ValAUC:0.9972 | Thr:0.49
  Ep  30/30 | TrLoss:0.0058 | ValLoss:0.1160 | ValF1:0.9916 | ValAUC:0.9972 | Thr:0.56

  [Abl1_CNN_Only_fold1] FINAL TEST RESULTS  (threshold=0.52)
  Accuracy  : 0.9819
  Precision : 0.0000
  Recall    : 0.0000
  F1 Score  : 0.0000
  Kappa     : -0.0003
  ROC-AUC   : 0.8261
  Train Time: 132.0s
  Infer Time: 1.23ms
  Model Size: 0.09MB
              precision    recall  f1-score   support

   No Stroke       0.98      1.00      0.99      6393
      Stroke       0.00      0.00      0.00       117

    accuracy                           0.98      6510
   macro avg       0.49      0.50      0.50      6510
weighted avg       0.96      0.98      0.97      6510

  Abl1_CNN_Only: F1=0.0000  AUC=0.8261  Thr=0.52
  Ep  10/30 | TrLoss

In [24]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/cerebral-stroke-predictionimbalaced-dataset/dataset.csv
/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv


In [25]:
# ============================================================
# CELL 13 — External / Unseen Data Test
# ============================================================
import os, time
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, roc_auc_score,
                             classification_report)

EXT_PATH = '/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv'

if os.path.exists(EXT_PATH):
    print("\n" + "="*60)
    print("  EXTERNAL DATASET TEST")
    print("="*60)

    ext_df = pd.read_csv(EXT_PATH)
    print("External data shape:", ext_df.shape)
    print(ext_df['stroke'].value_counts())

    ext_df = ext_df.drop(columns=['id'], errors='ignore')

    for col in cat_cols:
        if col in ext_df.columns:
            known = set(encoders[col].classes_)
            ext_df[col] = ext_df[col].astype(str).apply(
                lambda v: v if v in known else encoders[col].classes_[0])
            ext_df[col] = encoders[col].transform(ext_df[col])

    for col in FEATURE_COLS:
        if col not in ext_df.columns:
            ext_df[col] = 0

    ext_X        = ext_df[FEATURE_COLS].values
    ext_y        = ext_df['stroke'].values
    ext_X        = imputer.transform(ext_X)
    ext_X_scaled = scaler_C.transform(ext_X)

    # Load best model
    best_model_ext = build_model('full')
    checkpoint = torch.load('/kaggle/working/models/model_variantC.pt', weights_only=False)
    best_model_ext.load_state_dict(
        checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint
        else checkpoint)
    best_model_ext.to(device).eval()
    ext_threshold = checkpoint.get('threshold', 0.5) if isinstance(checkpoint, dict) else 0.5
    print(f"Loaded model with threshold: {ext_threshold:.2f}")

    # Inference
    ext_tensor = torch.FloatTensor(ext_X_scaled).unsqueeze(1).to(device)
    t0 = time.time()
    with torch.no_grad():
        ext_probs = torch.sigmoid(best_model_ext(ext_tensor).squeeze()).cpu().numpy()
    ext_inf_ms = (time.time() - t0) * 1000
    ext_preds  = (ext_probs >= ext_threshold).astype(int)

    print("\nClassification Report (External Data):")
    print(classification_report(ext_y, ext_preds, target_names=['No Stroke', 'Stroke']))

    res_ext = {
        'acc':           accuracy_score(ext_y, ext_preds),
        'prec':          precision_score(ext_y, ext_preds, zero_division=0),
        'rec':           recall_score(ext_y, ext_preds, zero_division=0),
        'f1':            f1_score(ext_y, ext_preds, zero_division=0),
        'kappa':         cohen_kappa_score(ext_y, ext_preds),
        'auc':           roc_auc_score(ext_y, ext_probs),
        'labels':        ext_y,
        'preds':         ext_preds,
        'probs':         ext_probs,
        'train_losses':  [],
        'val_losses':    [],
        'val_f1s':       [],
        'val_aucs':      [],
        'inf_time_ms':   ext_inf_ms,
        'train_time':    0,
        'model_size_mb': 0,
    }

    save_confusion_matrix(ext_y, ext_preds, 'External')
    save_roc_curve(ext_y, ext_probs, 'External')
    save_pr_curve(ext_y, ext_probs, 'External')
    print(f"✅ External test done | F1={res_ext['f1']:.4f} | AUC={res_ext['auc']:.4f}")

else:
    print("⚠️  External dataset not found. Add 'stroke-prediction-dataset' to your Kaggle inputs.")
    res_ext = None


  EXTERNAL DATASET TEST
External data shape: (5110, 12)
stroke
0    4861
1     249
Name: count, dtype: int64
Loaded model with threshold: 0.35

Classification Report (External Data):
              precision    recall  f1-score   support

   No Stroke       0.96      0.93      0.95      4861
      Stroke       0.21      0.33      0.25       249

    accuracy                           0.91      5110
   macro avg       0.59      0.63      0.60      5110
weighted avg       0.93      0.91      0.92      5110

✅ External test done | F1=0.2531 | AUC=0.8370


In [26]:
# ============================================================
# CELL 14 — GradCAM (1D) + SHAP on Proposed Model
# ============================================================

# ── Approach 1: 1D GradCAM via hooks on last CNN layer ──────
class GradCAM1D:
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, m, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, m, g_in, g_out):
        self.gradients = g_out[0].detach()

    def generate(self, x_tensor, class_idx=1):
        self.model.eval()
        self.model.zero_grad()
        x_tensor = x_tensor.clone().requires_grad_(True)
        out = self.model(x_tensor)
        score = out[:, 0]
        score.backward(torch.ones_like(score))
        # weights: mean over spatial dim
        weights = self.gradients.mean(dim=-1, keepdim=True)   # (B, C, 1)
        cam = (weights * self.activations).sum(dim=1)          # (B, seq_len)
        cam = torch.relu(cam)
        # Normalize per sample
        cam_min = cam.min(dim=1, keepdim=True)[0]
        cam_max = cam.max(dim=1, keepdim=True)[0]
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
        return cam.detach().cpu().numpy()


# Load best proposed model (Variant C)
proposed_model = build_model('full')
checkpoint = torch.load('/kaggle/working/models/model_variantC.pt', weights_only=False)
proposed_model.load_state_dict(checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint)
proposed_model.to(device).eval()

# Target: last CNNBlock (index 1) inside self.cnn
target_layer = proposed_model.cnn[1]
gradcam = GradCAM1D(proposed_model, target_layer)

# Sample 10 positive stroke cases from test set for visualization
pos_idx = np.where(y_teC == 1)[0][:10]
X_sample = torch.FloatTensor(X_teC[pos_idx]).unsqueeze(1).to(device)
cam_maps  = gradcam.generate(X_sample)   # (10, n_features)

# ── Plot GradCAM heatmap ──────────────────────────────────
mean_cam = cam_maps.mean(axis=0)   # average over samples

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# Bar chart of mean importance
axes[0].bar(FEATURE_NAMES, mean_cam, color=plt.cm.RdYlGn(mean_cam))
axes[0].set_title('GradCAM — Mean Feature Importance (Stroke Predictions)')
axes[0].set_ylabel('Normalized GradCAM Score')
axes[0].tick_params(axis='x', rotation=45)

# Heatmap over 10 samples
im = axes[1].imshow(cam_maps, aspect='auto', cmap='hot', interpolation='nearest')
axes[1].set_xticks(range(len(FEATURE_NAMES)))
axes[1].set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(len(pos_idx)))
axes[1].set_yticklabels([f'Sample {i+1}' for i in range(len(pos_idx))], fontsize=8)
axes[1].set_title('GradCAM Heatmap — 10 Stroke Samples')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('/kaggle/working/figures/gradcam_proposed.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print("✅ GradCAM saved.")

# ── Approach 2: SHAP DeepExplainer ──────────────────────────
print("\n🔵 Computing SHAP values...")
X_bg = torch.FloatTensor(X_trC_bal[:200]).unsqueeze(1).to(device)  # background
X_sh = torch.FloatTensor(X_teC[:100]).unsqueeze(1).to(device)      # explain these

explainer  = shap.DeepExplainer(proposed_model, X_bg)
shap_vals  = explainer.shap_values(X_sh)                            # (100, 1, n_features)
shap_arr   = np.array(shap_vals).squeeze()                          # (100, n_features)

fig, ax = plt.subplots(figsize=(9, 5))
shap.summary_plot(shap_arr, X_teC[:100], feature_names=FEATURE_NAMES,
                  show=False, plot_type='bar')
plt.title('SHAP Feature Importance — Proposed Model')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/shap_bar.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()

fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_arr, X_teC[:100], feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Beeswarm — Proposed Model')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print("✅ SHAP plots saved.")

RuntimeError: cudnn RNN backward can only be called in training mode

In [ ]:
# ============================================================
# CELL 15 — Final Summary Results Table
# ============================================================
print("\n" + "="*90)
print("  FINAL COMPREHENSIVE RESULTS SUMMARY")
print("="*90)

all_results = {
    'A: No Balancing':          res_A,
    'B: Full Balance (cGAN)':   res_B,
    'C: Balance Train Only':    res_C,
}
all_results.update(ablation_results)
if res_ext:
    all_results['External Test'] = res_ext

header = f"{'Variant':<30} | {'Acc':>6} | {'Prec':>6} | {'Rec':>6} | {'F1':>6} | {'Kappa':>6} | {'AUC':>6} | {'Train(s)':>9} | {'Infer(ms)':>10} | {'Size(MB)':>9}"
print(header)
print("-" * len(header))
for name, r in all_results.items():
    tr_t  = f"{r.get('train_time', 0):.1f}"
    inf_t = f"{r.get('inf_time_ms', 0):.2f}"
    sz    = f"{r.get('model_size_mb', 0):.2f}"
    print(f"  {name:<28} | {r['acc']:>6.4f} | {r['prec']:>6.4f} | {r['rec']:>6.4f} | "
          f"{r['f1']:>6.4f} | {r['kappa']:>6.4f} | {r['auc']:>6.4f} | "
          f"{tr_t:>9} | {inf_t:>10} | {sz:>9}")

# ── K-Fold mean±std table ──
print("\n── K-Fold 5-Fold Results (mean ± std) ──────────────────────────────")
print(f"{'Model':<30} | {'F1':^18} | {'AUC':^18} | {'Kappa':^18}")
print("-" * 90)
for name in ABLATION_CONFIGS:
    f1s   = kfold_scores[name]['f1']
    aucs  = kfold_scores[name]['auc']
    kaps  = kfold_scores[name]['kappa']
    print(f"  {name:<28} | {np.mean(f1s):.4f} ± {np.std(f1s):.4f}  "
          f"| {np.mean(aucs):.4f} ± {np.std(aucs):.4f}  "
          f"| {np.mean(kaps):.4f} ± {np.std(kaps):.4f}")

# ── Wilcoxon again cleanly ──
print("\n── Wilcoxon Test (Proposed vs Ablations, F1 over folds) ──────────")
proposed_f1 = kfold_scores['Abl4_Full_Proposed']['f1']
for name in ABLATION_CONFIGS:
    if name == 'Abl4_Full_Proposed': continue
    scores = kfold_scores[name]['f1']
    diff = np.array(proposed_f1) - np.array(scores)
    if np.all(diff == 0):
        print(f"  Proposed vs {name}: no difference (p=1.0)")
        continue
    try:
        stat, p = wilcoxon(proposed_f1, scores)
        sig = "✅ p<0.05 — Significant" if p < 0.05 else "❌ p≥0.05 — Not Significant"
        print(f"  Proposed vs {name:<28}: stat={stat:.4f}, p={p:.4f}  {sig}")
    except Exception as e:
        print(f"  Proposed vs {name}: {e}")

print("\n✅ All experiments complete!")
print(f"📁 Figures saved in: /kaggle/working/figures/")
print(f"📁 Models  saved in: /kaggle/working/models/")